Bronze to Silver Clean Pipeline

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re

storage_account = "azure1lake"
bronze_container = "bronze"
silver_container = "silver"

storage_key = "<paste your key>"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

DATASETS = {
    "customers": "AdventureWorks_Customers.csv",
    "products": "AdventureWorks_Products.csv",
    "product_categories": "AdventureWorks_Product_Categories.csv",
    "product_subcategories": "AdventureWorks_Product_Subcategories.csv",
    "calendar": "AdventureWorks_Calendar.csv",
    "territories": "AdventureWorks_Territories.csv",
    "returns": "AdventureWorks_Returns.csv",
    "sales_2015": "AdventureWorks_Sales_2015.csv",
    "sales_2016": "AdventureWorks_Sales_2016.csv",
    "sales_2017": "AdventureWorks_Sales_2017.csv"
}

In [0]:
def read_csv(file_name: str) -> DataFrame:
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(
            f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/{file_name}"
        )
    )

bronze_data = {}
for dataset_name, file_name in DATASETS.items():
    bronze_data[dataset_name] = read_csv(file_name)

sales_df = (
    bronze_data["sales_2015"]
    .unionByName(bronze_data["sales_2016"])
    .unionByName(bronze_data["sales_2017"])
)

customers_df = bronze_data["customers"]
products_df = bronze_data["products"]
returns_df = bronze_data["returns"]
calendar_df = bronze_data["calendar"]
territories_df = bronze_data["territories"]
product_categories_df = bronze_data["product_categories"]
product_subcategories_df = bronze_data["product_subcategories"]

print("✅ All Bronze datasets loaded successfully.")
display(dbutils.fs.ls(f"abfss://bronze@{storage_account}.dfs.core.windows.net/"))
display(customers_df.limit(5))
display(products_df.limit(5))
display(sales_df.limit(5))

✅ All Bronze datasets loaded successfully.


path,name,size,modificationTime
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Calendar.csv,AdventureWorks_Calendar.csv,9952,1783341869000
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Customers.csv,AdventureWorks_Customers.csv,1963594,1783341871000
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Product_Categories.csv,AdventureWorks_Product_Categories.csv,83,1783341869000
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Product_Subcategories.csv,AdventureWorks_Product_Subcategories.csv,637,1783341869000
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Products.csv,AdventureWorks_Products.csv,58122,1783341869000
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Returns.csv,AdventureWorks_Returns.csv,34587,1783341870000
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Sales_2015.csv,AdventureWorks_Sales_2015.csv,118594,1783341870000
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Sales_2016.csv,AdventureWorks_Sales_2016.csv,1083959,1783341872000
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Sales_2017.csv,AdventureWorks_Sales_2017.csv,1321941,1783341871000
abfss://bronze@azure1lake.dfs.core.windows.net/AdventureWorks_Territories.csv,AdventureWorks_Territories.csv,400,1783341870000


CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner
11000,MR.,JON,YANG,1966-04-08,M,M,jon24@adventure-works.com,"$90,000",2,Bachelors,Professional,Y
11001,MR.,EUGENE,HUANG,1965-05-14,S,M,eugene10@adventure-works.com,"$60,000",3,Bachelors,Professional,N
11002,MR.,RUBEN,TORRES,1965-08-12,M,M,ruben35@adventure-works.com,"$60,000",3,Bachelors,Professional,Y
11003,MS.,CHRISTY,ZHU,1968-02-15,S,F,christy12@adventure-works.com,"$70,000",0,Bachelors,Professional,N
11004,MRS.,ELIZABETH,JOHNSON,1968-08-08,S,F,elizabeth5@adventure-works.com,"$80,000",5,Bachelors,Professional,Y


ProductKey,ProductSubcategoryKey,ProductSKU,ProductName,ModelName,ProductDescription,ProductColor,ProductSize,ProductStyle,ProductCost,ProductPrice
214,31,HL-U509-R,"Sport-100 Helmet, Red",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Red,0,0,13.0863,34.99
215,31,HL-U509,"Sport-100 Helmet, Black",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Black,0,0,12.0278,33.6442
218,23,SO-B909-M,"Mountain Bike Socks, M",Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,M,U,3.3963,9.5
219,23,SO-B909-L,"Mountain Bike Socks, L",Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,L,U,3.3963,9.5
220,31,HL-U509-B,"Sport-100 Helmet, Blue",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Blue,0,0,12.0278,33.6442


OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2015-01-01,2001-09-21,SO45080,332,14657,1,1,1
2015-01-01,2001-12-05,SO45079,312,29255,4,1,1
2015-01-01,2001-10-29,SO45082,350,11455,9,1,1
2015-01-01,2001-11-16,SO45081,338,26782,6,1,1
2015-01-02,2001-12-15,SO45083,312,14947,10,1,1


In [0]:
def profile_dataframe(df, dataset_name):
    print("=" * 80)
    print(f"DATASET : {dataset_name}")
    print("=" * 80)
    columns = df.columns  # cache once to avoid repeated Analyze RPCs
    print(f"Rows    : {df.count()}")
    print(f"Columns : {len(columns)}")
    print("\nColumn Names")
    print(columns)
    print("\nSchema")
    df.printSchema()
    print("\nSample Data")
    display(df.limit(5))

def null_report(df):
    columns = df.columns  # cache once
    return df.select([
        count(when(col(column).isNull(), column)).alias(column)
        for column in columns
    ])

def duplicate_report(df):
    total_records = df.count()
    unique_records = df.dropDuplicates().count()
    duplicate_records = total_records - unique_records
    print(f"Total Records      : {total_records}")
    print(f"Unique Records     : {unique_records}")
    print(f"Duplicate Records  : {duplicate_records}")

def datatype_report(df):
    print("=" * 80)
    print("COLUMN NAME".ljust(35), "DATA TYPE")
    print("=" * 80)
    dtypes = df.dtypes  # cache once
    for column_name, datatype in dtypes:
        print(column_name.ljust(35), datatype)

def complete_profile(df, dataset_name):
    profile_dataframe(df, dataset_name)
    print("\n")
    print("=" * 80)
    print("NULL VALUE REPORT")
    print("=" * 80)
    display(null_report(df))
    print("\n")
    print("=" * 80)
    print("DUPLICATE REPORT")
    print("=" * 80)
    duplicate_report(df)
    print("\n")
    print("=" * 80)
    print("DATA TYPE REPORT")
    print("=" * 80)
    datatype_report(df)

In [0]:
complete_profile(customers_df, "Customers")
complete_profile(products_df, "Products")
complete_profile(sales_df, "Sales")
complete_profile(returns_df, "Returns")
complete_profile(calendar_df, "Calendar")
complete_profile(territories_df, "Territories")
complete_profile(product_categories_df, "Product Categories")
complete_profile(product_subcategories_df, "Product Subcategories")

DATASET : Customers
Rows    : 18148
Columns : 13

Column Names
['CustomerKey', 'Prefix', 'FirstName', 'LastName', 'BirthDate', 'MaritalStatus', 'Gender', 'EmailAddress', 'AnnualIncome', 'TotalChildren', 'EducationLevel', 'Occupation', 'HomeOwner']

Schema
root
 |-- CustomerKey: integer (nullable = true)
 |-- Prefix: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- BirthDate: date (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EmailAddress: string (nullable = true)
 |-- AnnualIncome: string (nullable = true)
 |-- TotalChildren: integer (nullable = true)
 |-- EducationLevel: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- HomeOwner: string (nullable = true)


Sample Data


CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner
11000,MR.,JON,YANG,1966-04-08,M,M,jon24@adventure-works.com,"$90,000",2,Bachelors,Professional,Y
11001,MR.,EUGENE,HUANG,1965-05-14,S,M,eugene10@adventure-works.com,"$60,000",3,Bachelors,Professional,N
11002,MR.,RUBEN,TORRES,1965-08-12,M,M,ruben35@adventure-works.com,"$60,000",3,Bachelors,Professional,Y
11003,MS.,CHRISTY,ZHU,1968-02-15,S,F,christy12@adventure-works.com,"$70,000",0,Bachelors,Professional,N
11004,MRS.,ELIZABETH,JOHNSON,1968-08-08,S,F,elizabeth5@adventure-works.com,"$80,000",5,Bachelors,Professional,Y




NULL VALUE REPORT


CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner
0,130,0,0,0,0,0,0,0,0,0,0,0




DUPLICATE REPORT
Total Records      : 18148
Unique Records     : 18148
Duplicate Records  : 0


DATA TYPE REPORT
COLUMN NAME                         DATA TYPE
CustomerKey                         int
Prefix                              string
FirstName                           string
LastName                            string
BirthDate                           date
MaritalStatus                       string
Gender                              string
EmailAddress                        string
AnnualIncome                        string
TotalChildren                       int
EducationLevel                      string
Occupation                          string
HomeOwner                           string
DATASET : Products
Rows    : 293
Columns : 11

Column Names
['ProductKey', 'ProductSubcategoryKey', 'ProductSKU', 'ProductName', 'ModelName', 'ProductDescription', 'ProductColor', 'ProductSize', 'ProductStyle', 'ProductCost', 'ProductPrice']

Schema
root
 |-- ProductKey: integer (nullabl

...



DUPLICATE REPORT
Total Records      : 4
Unique Records     : 4
Duplicate Records  : 0


DATA TYPE REPORT
COLUMN NAME                         DATA TYPE
ProductCategoryKey                  int
CategoryName                        string
DATASET : Product Subcategories
Rows    : 37
Columns : 3

Column Names
['ProductSubcategoryKey', 'SubcategoryName', 'ProductCategoryKey']

Schema
root
 |-- ProductSubcategoryKey: integer (nullable = true)
 |-- SubcategoryName: string (nullable = true)
 |-- ProductCategoryKey: integer (nullable = true)


Sample Data


ProductSubcategoryKey,SubcategoryName,ProductCategoryKey
1,Mountain Bikes,1
2,Road Bikes,1
3,Touring Bikes,1
4,Handlebars,2
5,Bottom Brackets,2




NULL VALUE REPORT


ProductSubcategoryKey,SubcategoryName,ProductCategoryKey
0,0,0




DUPLICATE REPORT
Total Records      : 37
Unique Records     : 37
Duplicate Records  : 0


DATA TYPE REPORT
COLUMN NAME                         DATA TYPE
ProductSubcategoryKey               int
SubcategoryName                     string
ProductCategoryKey                  int


In [0]:
def standardize_column_names(df):
    columns = df.columns  # cache once
    cleaned_columns = []
    for column in columns:
        column = column.strip()
        column = column.lower()
        column = re.sub(r"[ /()-]+", "_", column)
        column = re.sub(r"_+", "_", column)
        column = column.strip("_")
        cleaned_columns.append(column)
    return df.toDF(*cleaned_columns)

def trim_string_columns(df):
    schema_fields = df.schema.fields  # cache once
    string_columns = [
        field.name
        for field in schema_fields
        if isinstance(field.dataType, StringType)
    ]
    # Guard: withColumns({}) raises AssertionError on DataFrames with no string columns
    if not string_columns:
        return df
    return df.withColumns({column: trim(col(column)) for column in string_columns})

def remove_duplicates(df):
    before = df.count()
    df = df.dropDuplicates()
    after = df.count()
    print(f"Rows Before : {before}")
    print(f"Rows After  : {after}")
    print(f"Duplicates Removed : {before - after}")
    return df

def handle_missing_values(df):
    schema_fields = df.schema.fields  # cache once
    defaults = {}
    for field in schema_fields:
        if isinstance(field.dataType, StringType):
            defaults[field.name] = "Unknown"
        elif isinstance(field.dataType, IntegerType):
            defaults[field.name] = 0
        elif isinstance(field.dataType, DoubleType):
            defaults[field.name] = 0.0
    if defaults:
        df = df.fillna(defaults)  # single fillna call with all defaults
    return df

def validate_sales(df):
    columns = df.columns  # cache once
    if "salesamount" in columns:
        df = df.filter(col("salesamount") >= 0)
    return df

def clean_dataframe(df):
    print("=" * 80)
    print("Starting Cleaning Pipeline")
    print("=" * 80)
    df = standardize_column_names(df)
    df = trim_string_columns(df)
    df = remove_duplicates(df)
    df = handle_missing_values(df)
    df = validate_sales(df)
    print("=" * 80)
    print("Cleaning Completed")
    print("=" * 80)
    return df

In [0]:
def standardize_data_types(silver_data):
    # Use withColumns({...}) to apply multiple casts in a single plan node

    # annualincome arrives as a currency string from bronze, e.g. "$90,000 "
    # Step 1 — regexp_replace strips everything except digits: "$90,000 " → "90000"
    # Step 2 — cast to IntegerType: "90000" → 90000
    # Guard  — if the original value was null/blank, the cast produces null; fillna ensures 0
    silver_data["customers"] = (
        silver_data["customers"]
        .withColumns({
            "annualincome": regexp_replace(col("annualincome"), r"[^0-9]", "").cast(IntegerType()),
            "birthdate": to_date(col("birthdate"))
        })
        .fillna({"annualincome": 0})  # null-safe: blank-after-strip or original null → 0
    )

    silver_data["products"] = silver_data["products"].withColumns({
        "productcost": col("productcost").cast(DoubleType()),
        "productprice": col("productprice").cast(DoubleType())
    })

    silver_data["calendar"] = silver_data["calendar"].withColumns({
        "date": to_date(col("date"))
    })

    for table_name in ["sales_2015", "sales_2016", "sales_2017"]:
        silver_data[table_name] = silver_data[table_name].withColumns({
            "orderdate": to_date(col("orderdate")),
            "stockdate": to_date(col("stockdate"))
        })

    return silver_data

In [0]:
silver_data = {}
for dataset_name, dataframe in bronze_data.items():
    print()
    print(f"Cleaning {dataset_name}")
    silver_data[dataset_name] = clean_dataframe(dataframe)

silver_data = standardize_data_types(silver_data)

# --- AnnualIncome conversion check ---
# Bronze CSV stores AnnualIncome as a currency string: "$90,000 " (string, with $ and trailing space)
# standardize_data_types strips all non-digits and casts to IntegerType: 90000
# Verify the Silver type is integer and values are clean numbers (no $ or ,)
annualincome_dtype = dict(silver_data["customers"].dtypes).get("annualincome", "MISSING")
assert annualincome_dtype == "int", f"Expected int, got {annualincome_dtype}"
null_count = silver_data["customers"].filter(col("annualincome").isNull()).count()
print(f"✅ annualincome dtype = {annualincome_dtype}  |  null count = {null_count}")
print("Sample silver annualincome values (should be plain integers, no $ or ,):")

silver_data["customers"].printSchema()
display(silver_data["customers"].select("annualincome", "birthdate").limit(10))

silver_data["products"].printSchema()
display(silver_data["products"].select("productcost", "productprice").limit(10))

silver_data["calendar"].printSchema()
display(silver_data["calendar"].select("date").limit(10))

sales_clean = (
    silver_data["sales_2015"]
    .unionByName(silver_data["sales_2016"])
    .unionByName(silver_data["sales_2017"])
)

print(f"Total Sales Records : {sales_clean.count()}")


Cleaning customers
Starting Cleaning Pipeline
Rows Before : 18148
Rows After  : 18148
Duplicates Removed : 0
Cleaning Completed

Cleaning products
Starting Cleaning Pipeline
Rows Before : 293
Rows After  : 293
Duplicates Removed : 0
Cleaning Completed

Cleaning product_categories
Starting Cleaning Pipeline
Rows Before : 4
Rows After  : 4
Duplicates Removed : 0
Cleaning Completed

Cleaning product_subcategories
Starting Cleaning Pipeline
Rows Before : 37
Rows After  : 37
Duplicates Removed : 0
Cleaning Completed

Cleaning calendar
Starting Cleaning Pipeline
Rows Before : 912
Rows After  : 912
Duplicates Removed : 0
Cleaning Completed

Cleaning territories
Starting Cleaning Pipeline
Rows Before : 10
Rows After  : 10
Duplicates Removed : 0
Cleaning Completed

Cleaning returns
Starting Cleaning Pipeline
Rows Before : 1809
Rows After  : 1809
Duplicates Removed : 0
Cleaning Completed

Cleaning sales_2015
Starting Cleaning Pipeline
Rows Before : 2630
Rows After  : 2630
Duplicates Removed : 0

annualincome,birthdate
60000,1951-08-12
60000,1953-11-17
90000,1944-06-05
20000,1954-09-27
20000,1976-11-20
60000,1971-11-20
80000,1957-05-08
60000,1952-07-11
30000,1965-05-12
40000,1962-08-01


root
 |-- productkey: integer (nullable = false)
 |-- productsubcategorykey: integer (nullable = false)
 |-- productsku: string (nullable = false)
 |-- productname: string (nullable = false)
 |-- modelname: string (nullable = false)
 |-- productdescription: string (nullable = false)
 |-- productcolor: string (nullable = false)
 |-- productsize: string (nullable = false)
 |-- productstyle: string (nullable = false)
 |-- productcost: double (nullable = false)
 |-- productprice: double (nullable = false)



productcost,productprice
722.2568,1301.3636
8.2205,21.98
41.5723,53.99
144.5938,264.05
343.6496,539.99
413.1463,699.0982
413.1463,699.0982
24.9932,56.2909
1912.1544,3399.99
30.9334,74.99


root
 |-- date: date (nullable = true)



date
2015-03-09
2015-05-19
2016-03-01
2015-03-06
2016-04-25
2017-01-06
2015-04-09
2015-09-02
2015-12-22
2016-05-03


Total Sales Records : 56046


In [0]:
def write_to_silver(df, table_name):
    silver_path = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )
    print(f"✅ {table_name} written to Silver layer.")

for dataset_name, dataframe in silver_data.items():
    write_to_silver(dataframe, dataset_name)

display(dbutils.fs.ls(f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/"))

customers_silver = (
    spark.read
    .format("delta")
    .load(f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/customers")
)

display(customers_silver.limit(5))

✅ customers written to Silver layer.
✅ products written to Silver layer.
✅ product_categories written to Silver layer.
✅ product_subcategories written to Silver layer.
✅ calendar written to Silver layer.
✅ territories written to Silver layer.
✅ returns written to Silver layer.
✅ sales_2015 written to Silver layer.
✅ sales_2016 written to Silver layer.
✅ sales_2017 written to Silver layer.


path,name,size,modificationTime
abfss://silver@azure1lake.dfs.core.windows.net/calendar/,calendar/,0,1783413478000
abfss://silver@azure1lake.dfs.core.windows.net/customers/,customers/,0,1783425806000
abfss://silver@azure1lake.dfs.core.windows.net/product_categories/,product_categories/,0,1783413471000
abfss://silver@azure1lake.dfs.core.windows.net/product_subcategories/,product_subcategories/,0,1783413475000
abfss://silver@azure1lake.dfs.core.windows.net/products/,products/,0,1783413468000
abfss://silver@azure1lake.dfs.core.windows.net/returns/,returns/,0,1783413481000
abfss://silver@azure1lake.dfs.core.windows.net/sales/,sales/,0,1783414114000
abfss://silver@azure1lake.dfs.core.windows.net/sales_2015/,sales_2015/,0,1783584969000
abfss://silver@azure1lake.dfs.core.windows.net/sales_2016/,sales_2016/,0,1783584971000
abfss://silver@azure1lake.dfs.core.windows.net/sales_2017/,sales_2017/,0,1783584972000


customerkey,prefix,firstname,lastname,birthdate,maritalstatus,gender,emailaddress,annualincome,totalchildren,educationlevel,occupation,homeowner
11227,MR.,MARSHALL,CHAVEZ,1951-08-12,S,M,marshall35@adventure-works.com,60000,2,Partial College,Professional,Y
11316,MR.,LUKE,ALLEN,1953-11-17,M,M,luke52@adventure-works.com,60000,3,Graduate Degree,Management,Y
11415,MR.,RANDY,SHE,1944-06-05,S,M,randy25@adventure-works.com,90000,5,Partial College,Professional,N
11513,MRS.,ALYSSA,HOWARD,1954-09-27,S,F,alyssa38@adventure-works.com,20000,3,Partial High School,Manual,N
11605,MS.,HEIDI,SUBRAM,1976-11-20,S,F,heidi15@adventure-works.com,20000,0,High School,Manual,N


In [0]:
silver_data.pop("sales_2015", None)
silver_data.pop("sales_2016", None)
silver_data.pop("sales_2017", None)
silver_data["sales"] = sales_clean

for dataset_name, dataframe in silver_data.items():
    write_to_silver(dataframe, dataset_name)

for folder_name in ["sales_2015", "sales_2016", "sales_2017"]:
    try:
        dbutils.fs.rm(
            f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/{folder_name}",
            recurse=True
        )
    except Exception:
        pass

display(dbutils.fs.ls(f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/"))

def read_silver(table_name):
    return (
        spark.read
        .format("delta")
        .load(f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/{table_name}")
    )

customers_silver = read_silver("customers")
products_silver = read_silver("products")
sales_silver = read_silver("sales")
returns_silver = read_silver("returns")
calendar_silver = read_silver("calendar")
territories_silver = read_silver("territories")

✅ customers written to Silver layer.
✅ products written to Silver layer.
✅ product_categories written to Silver layer.
✅ product_subcategories written to Silver layer.
✅ calendar written to Silver layer.
✅ territories written to Silver layer.
✅ returns written to Silver layer.
✅ sales written to Silver layer.


path,name,size,modificationTime
abfss://silver@azure1lake.dfs.core.windows.net/calendar/,calendar/,0,1783413478000
abfss://silver@azure1lake.dfs.core.windows.net/customers/,customers/,0,1783425806000
abfss://silver@azure1lake.dfs.core.windows.net/product_categories/,product_categories/,0,1783413471000
abfss://silver@azure1lake.dfs.core.windows.net/product_subcategories/,product_subcategories/,0,1783413475000
abfss://silver@azure1lake.dfs.core.windows.net/products/,products/,0,1783413468000
abfss://silver@azure1lake.dfs.core.windows.net/returns/,returns/,0,1783413481000
abfss://silver@azure1lake.dfs.core.windows.net/sales/,sales/,0,1783414114000
abfss://silver@azure1lake.dfs.core.windows.net/territories/,territories/,0,1783413479000


In [0]:
silver_tables = {
    "customers":             read_silver("customers"),
    "products":              read_silver("products"),
    "product_categories":    read_silver("product_categories"),
    "product_subcategories": read_silver("product_subcategories"),
    "calendar":              read_silver("calendar"),
    "territories":           read_silver("territories"),
    "returns":               read_silver("returns"),
    "sales":                 read_silver("sales"),
}

print("=" * 70)
print(f"{'TABLE':<25} {'ROWS':>8}  {'COLS':>5}  KEY TYPE CHECKS")
print("=" * 70)

# Delta stores IntegerType as "int" — normalize aliases before comparing
type_checks = {
    "customers":  [("annualincome", "int"),    ("birthdate",    "date")],
    "products":   [("productcost",  "double"), ("productprice", "double")],
    "calendar":   [("date",         "date")],
    "sales":      [("orderdate",    "date"),   ("stockdate",    "date")],
}

all_ok = True
for name, df in silver_tables.items():
    row_count = df.count()
    col_count = len(df.columns)
    checks_str = ""
    if name in type_checks:
        dtypes_map = dict(df.dtypes)
        results = []
        for col_name, expected in type_checks[name]:
            actual = dtypes_map.get(col_name, "MISSING")
            ok = actual == expected
            if not ok:
                all_ok = False
            results.append(f"{col_name}={'OK' if ok else f'FAIL({actual}!={expected})'}")
        checks_str = "  ".join(results)
    print(f"{name:<25} {row_count:>8,}  {col_count:>5}  {checks_str}")

print("=" * 70)
print(f"\n{'✅ All type checks passed!' if all_ok else '❌ Some type checks failed — see above.'}")

# Confirm split sales folders are gone
silver_root = f"abfss://silver@{storage_account}.dfs.core.windows.net/"
silver_files = [f.name for f in dbutils.fs.ls(silver_root)]
for stale in ["sales_2015/", "sales_2016/", "sales_2017/"]:
    status = "❌ STILL PRESENT" if stale in silver_files else "✅ removed"
    print(f"{stale:<20} {status}")

print(f"\nSilver container: {len(silver_files)} table(s) → {sorted(silver_files)}")

TABLE                         ROWS   COLS  KEY TYPE CHECKS
customers                   18,148     13  annualincome=OK  birthdate=OK
products                       293     11  productcost=OK  productprice=OK
product_categories               4      2  
product_subcategories           37      3  
calendar                       912      1  date=OK
territories                     10      4  
returns                      1,809      4  
sales                       56,046      8  orderdate=OK  stockdate=OK

✅ All type checks passed!
sales_2015/          ✅ removed
sales_2016/          ✅ removed
sales_2017/          ✅ removed

Silver container: 8 table(s) → ['calendar/', 'customers/', 'product_categories/', 'product_subcategories/', 'products/', 'returns/', 'sales/', 'territories/']


Silver → Gold